In [0]:
df = spark.table("netflix_titles")
df.groupBy("TYPE").count().display()

TYPE,count
Movie,6131
TV Show,2676
null,1
William Wyler,1


In [0]:
df.show(4)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

In [0]:
df.filter(df.DIRECTOR.isNull()).count()

2636

In [0]:
df.groupBy("RATING") \
    .count() \
    .orderBy("count", ascending=False) \
    .display()

RATING,count
TV-MA,3207
TV-14,2160
TV-PG,862
R,799
PG-13,490
TV-Y7,334
TV-Y,307
PG,287
TV-G,220
NR,80


In [0]:
df.groupBy("listed_in").count().orderBy("count", ascending=False).show(10,truncate=False)

+------------------------------------------------+-----+
|listed_in                                       |count|
+------------------------------------------------+-----+
|Dramas, International Movies                    |362  |
|Documentaries                                   |359  |
|Stand-Up Comedy                                 |334  |
|Comedies, Dramas, International Movies          |274  |
|Dramas, Independent Movies, International Movies|252  |
|Kids' TV                                        |220  |
|Children & Family Movies                        |215  |
|Children & Family Movies, Comedies              |201  |
|Documentaries, International Movies             |186  |
|Dramas, International Movies, Romantic Movies   |180  |
+------------------------------------------------+-----+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import explode, split, trim

df.withColumn(
    "GENRE",
    explode(split(df["LISTED_IN"], ","))
).withColumn(
    "GENRE",
    trim("GENRE")
).groupBy("GENRE") \
 .count() \
 .orderBy("count", ascending=False) \
 .limit(15) \
 .display()

GENRE,count
International Movies,2752
Dramas,2427
Comedies,1674
International TV Shows,1351
Documentaries,868
Action & Adventure,859
TV Dramas,763
Independent Movies,756
Children & Family Movies,641
Romantic Movies,616


In [0]:
from pyspark.sql.functions import col, try_to_date, year, month, count, trim

df = df.withColumn(
    "added_date",
    try_to_date(trim(col("date_added")), "MMMM d, yyyy")
)

df = df.withColumn(
    "year", year(col("added_date"))
).withColumn(
    "month", month(col("added_date"))
)

df.filter(col("added_date").isNotNull()) \
  .groupBy("year", "month") \
  .count() \
  .orderBy("year", "month") \
  .show(50)

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2008|    1|    1|
|2008|    2|    1|
|2009|    5|    1|
|2009|   11|    1|
|2010|   11|    1|
|2011|    5|    1|
|2011|    9|    1|
|2011|   10|   11|
|2012|    2|    1|
|2012|   11|    1|
|2012|   12|    1|
|2013|    3|    1|
|2013|    8|    1|
|2013|    9|    2|
|2013|   10|    3|
|2013|   11|    2|
|2013|   12|    2|
|2014|    1|    2|
|2014|    2|    2|
|2014|    4|    2|
|2014|    6|    1|
|2014|    7|    1|
|2014|    8|    1|
|2014|    9|    1|
|2014|   10|    4|
|2014|   11|    4|
|2014|   12|    6|
|2015|    1|    1|
|2015|    2|    4|
|2015|    3|    5|
|2015|    4|    5|
|2015|    5|    6|
|2015|    6|    6|
|2015|    7|    7|
|2015|    8|    2|
|2015|    9|    7|
|2015|   10|   14|
|2015|   11|    4|
|2015|   12|   21|
|2016|    1|   43|
|2016|    2|   15|
|2016|    3|   17|
|2016|    4|   22|
|2016|    5|   13|
|2016|    6|   18|
|2016|    7|   28|
|2016|    8|   34|
|2016|    9|   48|
|2016|   10|   51|
|2016|   11|

In [0]:
from pyspark.sql.functions import col, regexp_extract, avg, round

df.filter(col("TYPE") == "TV Show") \
    .withColumn(
        "SEASONS",
        regexp_extract(col("DURATION"), r"(\d+)", 1).cast("int")
    ) \
    .select(
        round(avg("SEASONS"), 2).alias("AVERAGE_SEASONS")
    ) \
    .show()

+---------------+
|AVERAGE_SEASONS|
+---------------+
|           1.76|
+---------------+



In [0]:
from pyspark.sql.functions import col

df.filter(
    col("TITLE").contains("Christmas") |
    col("DESCRIPTION").contains("Christmas")
).select(
    "SHOW_ID",
    "TITLE",
    "TYPE",
    "DESCRIPTION"
).display()

SHOW_ID,TITLE,TYPE,DESCRIPTION
s657,Love Is a Story,Movie,"Two young artists, a violin player and an actress, fight to achieve their dreams and keep their relationship together during Christmastime."
s1517,Home for Christmas,TV Show,"Tired of the constant comments on her relationship status, perpetually single Johanne starts a 24-day hunt for a boyfriend to bring home for Christmas."
s1524,An Unremarkable Christmas,Movie,An accountant and aspiring magician invites his boss to spend Christmas with his family — unaware that he's one of Colombia's most-wanted criminals.
s1536,How To Ruin Christmas,TV Show,Prodigal daughter Tumi tries to make things right after completely ruining what should have been her sister's picture-perfect Christmas wedding.
s1551,A California Christmas,Movie,"With his carefree lifestyle on the line, a wealthy charmer poses as a ranch hand to get a hardworking farmer to sell her family’s land before Christmas."
s1557,A Trash Truck Christmas,Movie,"When Santa crash-lands in the junkyard on Christmas Eve, Hank, Trash Truck and their animal friends all have a hand in rescuing the holiday for everyone."
s1589,Mighty Express: A Mighty Christmas,Movie,The Mighty Express welcomes a new train and speeds into a snowstorm to deliver Santa’s presents and save Christmas for all the kids in Tracksville.
s1592,Captain Underpants Mega Blissmas,Movie,Christmas gets weird — really weird — after George and Harold go back in time to change up a few of their beloved holiday's traditions.
s1593,Christmas Crossfire,Movie,"A man foils an attempted murder, then flees the crew of would-be killers along with their intended target as a woman he's just met tries to find him."
s1602,Just Another Christmas,Movie,"Stuck in a time loop where it's forever Christmas, a family man who hates the holiday starts to learn valuable lessons about what's important in life."


In [0]:
from pyspark.sql.functions import col, lower

df.filter(
    (col("TYPE") == "Movie") &
    (lower(col("CAST")).contains("ryan reynolds"))
).select(
    "TITLE",
    "RELEASE_YEAR",
    "CAST",
    "DESCRIPTION"
).orderBy(
    col("RELEASE_YEAR").desc()
).display()

TITLE,RELEASE_YEAR,CAST,DESCRIPTION
6 Underground,2019,"Ryan Reynolds, Mélanie Laurent, Corey Hawkins, Dave Franco, Adria Arjona, Manuel Garcia-Rulfo, Ben Hardy, Lior Raz, Payman Maadi, Yuri Kolokolnikov, Kim Kold","After faking his death, a tech billionaire recruits a team of international operatives for a bold and bloody mission to take down a brutal dictator."
Betty White: First Lady of Television,2018,"Betty White, Valerie Bertinelli, Georgia Engel, Tina Fey, Valerie Harper, Jennifer Love Hewitt, Carl Reiner, Ryan Reynolds, Alex Trebek",This documentary on actress and television producer Betty White traces her decades-long career as a woman breaking new ground in entertainment.
Mississippi Grind,2015,"Ryan Reynolds, Ben Mendelsohn, Sienna Miller, Analeigh Tipton, Alfre Woodard, James Toback, Robin Weigert",A gambler mired in debt thinks another cardsharp may just be his good luck charm as the two head for New Orleans to buy into a high-stakes poker game.
Selfless,2015,"Ryan Reynolds, Natalie Martinez, Matthew Goode, Ben Kingsley, Victor Garber, Derek Luke, Michelle Dockery, Melora Hardin, Brendan McCarthy, Sam Page","A dying tycoon has his consciousness transplanted into the body of a healthy young man, only to make a troubling discovery about the body's origin."
The Captive,2014,"Ryan Reynolds, Scott Speedman, Rosario Dawson, Mireille Enos, Kevin Durand, Alexia Fast, Christine Horne, Arsinée Khanjian, Ian Matthews, Bruce Greenwood, Aidan Shipley","Eight years after their daughter's abduction tore them apart, her parents receive enigmatic clues from the kidnapper hinting that she's still alive."
The Croods,2013,"Nicolas Cage, Emma Stone, Ryan Reynolds, Catherine Keener, Cloris Leachman, Clark Duke, Chris Sanders, Randy Thom","When an earthquake obliterates their cave, an unworldly prehistoric family is forced to journey through unfamiliar terrain in search of a new home."
Turbo,2013,"Ben Schwartz, Bill Hader, Luis Guzmán, Snoop Dogg, Paul Giamatti, Ryan Reynolds, Maya Rudolph, Ken Jeong, Samuel L. Jackson, Michael Peña, Michelle Rodriguez, Richard Jenkins, Kurtwood Smith, Luis Guzman",A speed-obsessed snail who dreams of being the world's greatest race car driver gets his chance when an accident imbues him with high-octane speed.
Safe House,2012,"Denzel Washington, Ryan Reynolds, Vera Farmiga, Brendan Gleeson, Sam Shepard, Rubén Blades, Nora Arnezeder, Robert Patrick, Liam Cunningham, Joel Kinnaman",Young CIA operative Matt Weston must get a dangerous criminal out of an agency safe house that's come under attack and get him to a securer location.
Green Lantern,2011,"Ryan Reynolds, Blake Lively, Peter Sarsgaard, Mark Strong, Tim Robbins, Jay O. Sanders, Taika Waititi, Angela Bassett",Test pilot Hal Jordan harnesses glowing new powers for good when he wears an otherworldly ring and helps an intergalactic force stop a powerful threat.
Just Friends,2005,"Anna Faris, Ty Olsson, Julie Hagerty, Amy Smart, Chris Marquette, Fred Ewanuick, Amy Matysio, Chris Klein, Ryan Reynolds, Stephen Root","Years after high school, a once-portly nice guy comes home for the holidays and reunites with a former crush he still has feelings for."


In [0]:
df.filter(
    (col("RELEASE_YEAR") >= 2020) &
    (col("RELEASE_YEAR") <= 2021)
).count()

1545

In [0]:
df.filter(
    col("RELEASE_YEAR").isin(2019, 2020, 2021)
).groupBy("RELEASE_YEAR") \
 .count() \
 .orderBy("RELEASE_YEAR") \
 .display()

RELEASE_YEAR,count
2019,1030
2020,953
2021,592


In [0]:
df.filter(
    (regexp_extract(col("DURATION"), r"(\d+)", 1).cast("int") <= 120)
).select(
    "TITLE","TYPE",
    "RELEASE_YEAR",
    "DURATION",
    "RATING",
    "DESCRIPTION"
).orderBy("DURATION").display()

TITLE,TYPE,RELEASE_YEAR,DURATION,RATING,DESCRIPTION
Exhibit A,TV Show,2019,1 Season,TV-MA,This true crime series shows how innocent people have been convicted with dubious forensic techniques and tools such as touch DNA and cadaver dogs.
Sweet Tooth,TV Show,2021,1 Season,TV-14,"On a perilous adventure across a post-apocalyptic world, a lovable boy who's half-human and half-deer searches for a new beginning with a gruff protector."
"O-Negative, Love Can’t Be Designed",TV Show,2016,1 Season,TV-MA,"Five schoolmates who share a blood type navigate the vagaries of friendship, love and university life."
Civilizations,TV Show,2018,1 Season,TV-14,"Expanding Kenneth Clark's 1969 ""Civilisation"" series for a new generation, this reboot explores the art of cultures around the world, throughout time."
Here to Heart,TV Show,2018,1 Season,TV-14,Wen Nuan signs on to work as the executive assistant to ex Zhan Nanxian at his tech firm. Helping him at work just may lead her back into his heart.
Elite Short Stories: Carla Samuel,TV Show,2021,1 Season,TV-MA,"Samuel attempts a big, romantic gesture at the airport in order to persuade Carla not to board her flight to London."
Chicago Typewriter,TV Show,2017,1 Season,TV-14,A veterinarian and two writers have a mysterious connection to Korean independence fighters from the Japanese colonial period.
Seis Manos,TV Show,2019,1 Season,TV-MA,"Orphans raised by a martial arts master are plunged into a mystery involving demonic powers, drug cartels, ancient rituals and blood sacrifice."
Vampires,TV Show,2020,1 Season,TV-MA,"A Paris teen who's half human, half vampire grapples with her emerging powers and family turmoil as she's pursued by a secret vampire community."
The Club,TV Show,2019,1 Season,TV-MA,"A band of misfit rich kids in Mexico strike out on their own selling MDMA and quickly run into trouble with other narcos, the law and their families."


In [0]:
from pyspark.sql.functions import col, regexp_extract

df.filter(
    (col("TYPE") == "Movie") &
    (regexp_extract(col("DURATION"), r"(\d+)", 1).cast("int") <= 120)
).select(
    "TITLE",
    "RELEASE_YEAR",
    "DURATION",
    "RATING",
    "DESCRIPTION"
).orderBy("DURATION").display()

TITLE,RELEASE_YEAR,DURATION,RATING,DESCRIPTION
American Factory: A Conversation with the Obamas,2019,10 min,TV-G,Barack and Michelle Obama talk with directors Steven Bognar and Julia Reichert about the documentary and the importance of storytelling.
Oversize Cops,2017,100 min,TV-MA,"While investigating a bank robbery, a squad of overweight police officers undergoes rigorous training to shed pounds."
My Horrible Grandma,2012,100 min,TV-MA,"As a man fumbles to keep things in order, his assertive grandmother tries to straighten out his life when she returns to live with him."
Black '47,2018,100 min,R,"When the Great Famine ravages his beloved country, a battle-hardened Irishman deserts the British Empire and exacts revenge on the tyrants responsible."
Reversing Roe,2018,100 min,TV-14,"Through interviews with abortion rights supporters and opponents, this film lifts the lid on a decades-long political campaign to overturn Roe v. Wade."
Regatta,2015,100 min,TV-MA,Inside one of Cairo's most impoverished areas exists a place where criminals control everything and everyone.
Winter's Bone,2010,100 min,R,"In this unflinching noir drama, a resilient teen goes on the trail of her missing, drug-dealing father when his absence threatens the family's safety."
Fractured,2019,100 min,TV-MA,"After his wife and injured daughter disappear from an ER, a man conducts a panicked search and becomes convinced the hospital is hiding something."
Asu Mare 2,2015,100 min,TV-14,To win the heart of his true love – a high-society girl – a hapless commoner plots to change his identity and avoid his embarrassing friends.
Virunga,2014,100 min,TV-MA,The Oscar-nominated true story of the rangers risking their lives to save Africa's most precious national park and its endangered gorillas.


In [0]:
from pyspark.sql.functions import col, split, avg, round
result = (
    df
    .filter(col("type") == "TV Show")
    .filter(col("duration").like("%Season%"))
    .withColumn(
        "seasons",
        split(col("duration"), " ")[0].cast("int")
    )
    .agg(
        round(avg("seasons"), 2).alias("average_seasons")
    )
)
result.show()

+---------------+
|average_seasons|
+---------------+
|           1.76|
+---------------+

